## Re-parse credential-contaminated author_names rows

One-off backfill. `CreateAuthorNames` is incremental (`LEFT ANTI JOIN` against existing `author_names`), so parser-code changes don't retroactively apply to the ~222M rows already in the table. This notebook deletes the rows whose OLD parse put a credential acronym in `first`, `last`, or `middle` so the next `CreateAuthorNames` run re-parses them with the updated credential-stripping logic.

**Scope:** ~804K rows (0.36% of 222M).

**Sequence:**
1. Verify counts in Cell 2
2. Spot-check bad parses in Cell 3
3. Run Cell 4 (DELETE)
4. Run the `CreateAuthorNames` job — it will re-parse the deleted raws via the new logic
5. Re-run Cell 5 to verify the same sample of raws now parses cleanly
6. Re-run `CreateAuthors` to refresh `authors_for_matching` / downstream aggregates

**Not in scope:** updating existing `work_authors.author_id` assignments. Those were made by MatchAuthors at ingest time and aren't directly tied to `author_names.parsed_name`. The re-parse improves diagnostic queries and future matches; existing assignments stay until a Phase 1 re-run.

**Safety:** Delta `DELETE` only; the rows will be re-created by the next `CreateAuthorNames` run. Credential list matches `_NP_CREDENTIAL_ACRONYMS` + `_NP_DOTTED_ONLY_CREDS` in `CreateAuthorNames` cell 4.

### Cell 1: Verify affected row count by slot

In [ ]:
SELECT
  SUM(CASE WHEN lower(parsed_name.first) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  ) THEN 1 ELSE 0 END) AS first_is_cred,
  SUM(CASE WHEN lower(parsed_name.last) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  ) THEN 1 ELSE 0 END) AS last_is_cred,
  SUM(CASE WHEN lower(parsed_name.middle) RLIKE '(^|\\s)(phd|dphil|drph|scd|edd|psyd|pharmd|mbbs|mbchb|dvm|dds|dmd|dnp|dpt|msc|mph|mscn|msn|mpp|mhs|llm|jd|lcsw|facs|facp|facog|faap|fccp|faan|frcp|frcs|frcpc|frcog|mrcp|mrcs|fcps|rn|crna|fnp|md|do|ms|ma|mba|mfa|ba|bs)($|\\s)' THEN 1 ELSE 0 END) AS middle_has_cred,
  COUNT(*) AS total_rows
FROM openalex.authors.author_names

### Cell 2: Total affected (union) — the row count that will be deleted

In [ ]:
SELECT COUNT(*) AS rows_to_delete
FROM openalex.authors.author_names
WHERE lower(parsed_name.first) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.last) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.middle) RLIKE '(^|\\s)(phd|dphil|drph|scd|edd|psyd|pharmd|mbbs|mbchb|dvm|dds|dmd|dnp|dpt|msc|mph|mscn|msn|mpp|mhs|llm|jd|lcsw|facs|facp|facog|faap|fccp|faan|frcp|frcs|frcpc|frcog|mrcp|mrcs|fcps|rn|crna|fnp|md|do|ms|ma|mba|mfa|ba|bs)($|\\s)'

### Cell 3: Spot-check — 30 current bad parses (save these raws to verify post-rerun)

In [ ]:
SELECT raw_author_name,
       parsed_name.first AS first,
       parsed_name.middle AS middle,
       parsed_name.last AS last
FROM openalex.authors.author_names
WHERE lower(parsed_name.first) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.last) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
ORDER BY RAND()
LIMIT 30

### Cell 4: Delete the credential-contaminated rows

**WARNING**: destructive. Run Cells 1–3 first and confirm scope looks right.

Delta `DELETE` — rows will be re-created by the next `CreateAuthorNames` run via its `LEFT ANTI JOIN` against `locations_mapped` / `openalex_authors` / `work_authors`.

In [ ]:
DELETE FROM openalex.authors.author_names
WHERE lower(parsed_name.first) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.last) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.middle) RLIKE '(^|\\s)(phd|dphil|drph|scd|edd|psyd|pharmd|mbbs|mbchb|dvm|dds|dmd|dnp|dpt|msc|mph|mscn|msn|mpp|mhs|llm|jd|lcsw|facs|facp|facog|faap|fccp|faan|frcp|frcs|frcpc|frcog|mrcp|mrcs|fcps|rn|crna|fnp|md|do|ms|ma|mba|mfa|ba|bs)($|\\s)'

### Cell 5: Post-delete verification

Expect 0 rows remaining with credential in `first`/`last`/`middle` (before the re-parse adds new rows back).

**Next step:** run the `Authors` job / `CreateAuthorNames` notebook to re-parse the deleted raws. Then re-run this cell — expect 0 rows again (credentials properly stripped by the new logic). Then re-run `CreateAuthors` to refresh `authors_for_matching`.

In [ ]:
SELECT COUNT(*) AS remaining_contaminated
FROM openalex.authors.author_names
WHERE lower(parsed_name.first) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )
   OR lower(parsed_name.last) IN (
    'phd','dphil','drph','scd','edd','psyd','pharmd','mbbs','mbchb','dvm','dds','dmd','dnp','dpt',
    'msc','mph','mscn','msn','mpp','mhs','llm','jd','lcsw',
    'facs','facp','facog','faap','fccp','faan','frcp','frcs','frcpc','frcog','mrcp','mrcs','fcps',
    'rn','crna','fnp','md','do','ms','ma','mba','mfa','ba','bs'
  )